In [ ]:
import pandas as pd

In [ ]:
heart_df = pd.read_csv('./data/heart.csv')
heart_df['HeartDisease'] = heart_df['HeartDisease'].map({'No':0,'Yes':1})
heart_df

,HeartDisease,BMI,Smoking,AlcoholDrinking,Stroke,PhysicalHealth,MentalHealth,DiffWalking,Sex,AgeCategory,Race,Diabetic,PhysicalActivity,GenHealth,SleepTime,Asthma,KidneyDisease,SkinCancer
0,0,16.60,Yes,No,No,3,30,No,Female,55-59,White,Yes,Yes,Very good,5,Yes,No,Yes
1,0,20.34,No,No,Yes,0,0,No,Female,80 or older,White,No,Yes,Very good,7,No,No,No
2,0,26.58,Yes,No,No,20,30,No,Male,65-69,White,Yes,Yes,Fair,8,Yes,No,No
3,0,24.21,No,No,No,0,0,No,Female,75-79,White,No,No,Good,6,No,No,Yes
4,0,23.71,No,No,No,28,0,Yes,Female,40-44,White,No,Yes,Very good,8,No,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319790,1,27.41,Yes,No,No,7,0,Yes,Male,60-64,Hispanic,Yes,No,Fair,6,Yes,No,No
319791,0,29.84,Yes,No,No,0,0,No,Male,35-39,Hispanic,No,Yes,Very good,5,Yes,No,No
319792,0,24.24,No,No,No,0,0,No,Female,45-49,Hispanic,No,Yes,Good,6,No,No,No
319793,0,32.81,No,No,No,0,0,No,Female,25-29,Hispanic,No,No,Good,12,No,No,No


Voting classifier with hard voting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# ---------------------------
# 1. Load data
# ---------------------------
df = pd.read_csv("./data/heart_2020_cleaned.csv")
df['HeartDisease'] = df['HeartDisease'].map({'No':0, 'Yes':1})

X = df.drop("HeartDisease", axis=1)  
# ---------------------------
# 2. Column types
# ---------------------------
categorical_cols = X.select_dtypes(include=['object']).columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns

# ---------------------------
# 3. Preprocessing
# ---------------------------
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])

# ---------------------------
# 4. Train-test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ---------------------------
# 5. Handle imbalance with SMOTE
# ---------------------------
smote = SMOTE(random_state=42)

# ---------------------------
# 6. Define base models with best parameters observed
# ---------------------------
rf = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    random_state=42,
    class_weight='balanced'
)

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5
)

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    use_label_encoder=False,
    scale_pos_weight=(sum(y_train==0)/sum(y_train==1)),
    random_state=42
)

lr = LogisticRegression(
    max_iter=5000,
    class_weight='balanced',
    solver='liblinear'
)

# ---------------------------
# 7. Voting Classifier
# ---------------------------
voting_clf = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('xgb', xgb), ('lr', lr)],
    voting='hard',
    n_jobs=-1,
    weights=[1,2,2,1]  # give more weight to boosting models
)

# ---------------------------
# 8. Full pipeline
# ---------------------------
pipeline = ImbPipeline([
    ('preprocess', preprocessor),
    ('smote', smote),
    ('voting', voting_clf)
])

# ---------------------------
# 9. Train
# ---------------------------
pipeline.fit(X_train, y_train)
 
# ---------------------------                      nb hg
# 10. Predict probabilities and classes
# ---------------------------
y_pred = pipeline.predict(X_test)  # risk score (0-1)

# ---------------------------
# 11. Evaluate
# ---------------------------
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Voting Classifier with soft voting (SMOTE)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# ---------------------------
# 1. Load data
# ---------------------------
df = pd.read_csv("./data/heart_2020_cleaned.csv")
df['HeartDisease'] = df['HeartDisease'].map({'No':0, 'Yes':1})

X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

# ---------------------------
# 2. Column types
# ---------------------------
categorical_cols = X.select_dtypes(include=['object']).columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns

# ---------------------------
# 3. Preprocessing
# ---------------------------
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])

# ---------------------------
# 4. Train-test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ---------------------------
# 5. Handle imbalance with SMOTE
# ---------------------------
smote = SMOTE(random_state=42)

# ---------------------------
# 6. Define base models with best parameters observed
# ---------------------------
rf = RandomForestClassifier(
    n_estimators=250,
    max_depth=12,
    random_state=42,
    class_weight='balanced'
)

gb = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5
)

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    use_label_encoder=False,
    scale_pos_weight=(sum(y_train==0)/sum(y_train==1)),
    random_state=42
)

lr = LogisticRegression(
    max_iter=5000,
    class_weight='balanced',
    solver='liblinear'
)

# ---------------------------
# 7. Voting Classifier
# ---------------------------
voting_clf = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('xgb', xgb), ('lr', lr)],
    voting='soft',
    n_jobs=-1,
    weights=[1,2,2,1]  # give more weight to boosting models
)

# ---------------------------
# 8. Full pipeline
# ---------------------------
pipeline = ImbPipeline([
    ('preprocess', preprocessor),
    ('smote', smote),
    ('voting', voting_clf)
])

# ---------------------------
# 9. Train
# ---------------------------
pipeline.fit(X_train, y_train)

# ---------------------------
# 10. Predict probabilities and classes
# ---------------------------
y_proba = pipeline.predict_proba(X_test)[:,1]  # risk score (0-1)

# Tunable threshold to balance recall vs precision
threshold = 0.3
y_pred = (y_proba >= threshold).astype(int)

# ---------------------------
# 11. Evaluate
# ---------------------------
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ---------------------------
# 12. Risk categories for app
# ---------------------------
def get_risk_category(prob):
    if prob < 0.2:
        return 'Low'
    elif prob < 0.5:
        return 'Moderate'
    else:
        return 'High'

risk_category = [get_risk_category(p) for p in y_proba]

# ---------------------------
# 13. Create final output DataFrame
# ---------------------------
df_result = X_test.copy()
df_result['RiskScore'] = y_proba
df_result['RiskCategory'] = risk_category
df_result['PredictedLabel'] = y_pred
df_result['TrueLabel'] = y_test.values

df_result.head()


Stacking Classifier

In [ ]:
# === Stacking classifier (XGB + GB + KNN + RF) + SHAP + threshold tuning ===
# Requirements: scikit-learn, xgboost, imbalanced-learn, shap, matplotlib, seaborn, joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
#import shap
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve
)
from sklearn.inspection import permutation_importance

# ---------------------------
# 0) Data - replace if needed
# ---------------------------
# You said you already have the dataset loaded; I'll assume `heart_df` exists with 'HeartDisease' target.
df = heart_df.copy()                      # <--- ensure heart_df is defined
TARGET = "HeartDisease"                   # adjust if different

X = df.drop(columns=[TARGET])
y = df['HeartDisease']

# train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ---------------------------
# 1) Columns separation
# ---------------------------
num_cols = X_train.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object','category']).columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

# ---------------------------
# 2) Preprocessor
# ---------------------------
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])

# ---------------------------
# 3) Best-guess parameters (no GridSearch)
# ---------------------------
# Compute scale_pos_weight for XGBoost (useful if you want to pass it)
pos = y_train.sum()
neg = len(y_train) - pos
scale_pos_weight = (neg / pos) if pos > 0 else 1.0
print("scale_pos_weight:", round(scale_pos_weight, 3))

# XGBoost params (robust for tabular clinical data)
xgb_clf = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_weight=3,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)

# Gradient Boosting (sklearn)
gb_clf = GradientBoostingClassifier(
    n_estimators=250,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.9,
    random_state=42
)

# KNN (sensitive to scaling)
knn_clf = KNeighborsClassifier(n_neighbors=15, weights="distance", p=2)

# Random Forest
rf_clf = RandomForestClassifier(
    n_estimators=400,
    max_depth=8,
    min_samples_leaf=2,
    class_weight='balanced_subsample',
    n_jobs=-1,
    random_state=42
)

# Stacking final estimator
final_estimator = LogisticRegression(max_iter=2000, class_weight='balanced')

# ---------------------------
# 4) Build stacking classifier inside imbalanced pipeline
# ---------------------------
estimators = [
    ("xgb", xgb_clf),
    ("gb", gb_clf),
    ("knn", knn_clf),
    ("rf", rf_clf)
]

stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=final_estimator,
    stack_method="predict_proba",  # use predict_proba outputs as meta-features
    n_jobs=-1,
    passthrough=False
)

# Pipeline: preprocess -> SMOTE -> stacking
pipe = ImbPipeline([
    ("preprocess", preprocessor),
    ("smote", SMOTE(random_state=42)),   # switch to SMOTEENN() if you prefer
    ("model", stacking)
])

# ---------------------------
# 5) Fit the pipeline
# ---------------------------
pipe.fit(X_train, y_train)

# ---------------------------
# 6) Predict probabilities on test
# ---------------------------
probs_test = pipe.predict_proba(X_test)[:, 1]
preds_default = (probs_test >= 0.19).astype(int)

acc = accuracy_score(y_test, preds_default)
prec = precision_score(y_test, preds_default, zero_division=0)
rec = recall_score(y_test, preds_default)
f1 = f1_score(y_test, preds_default)
auc = roc_auc_score(y_test, probs_test)
tn, fp, fn, tp = confusion_matrix(y_test, preds_default).ravel()

print(f"Accuracy: {acc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}  AUC: {auc:.4f}")
print(f"TP={tp}, FN={fn}, FP={fp}, TN={tn}")
print("\nCLASSIFICATION REPORT:\n", classification_report(y_test, probs_test))

NameError: name 'heart_df' is not defined